In [69]:
import requests
import re
import uuid

In [2]:
SPARQL_ENDPOINT = 'http://localhost:7001'
CRISP_NAMESPACE = 'http://crispy.ai.wu.ac.at/crisp/'
SOGO_NAMESPACE = "http://example.org/sogo#"

TYPE_CONVERTERS = {
	"http://www.w3.org/2001/XMLSchema#int": int,
	"http://www.w3.org/2001/XMLSchema#integer": int,
	"http://www.w3.org/2001/XMLSchema#float": float,
	"http://www.w3.org/2001/XMLSchema#double": float,
	"http://www.w3.org/2001/XMLSchema#boolean": lambda x: x.lower() == "true",
	"http://www.w3.org/2001/XMLSchema#string": str,
}

In [97]:
class SogoScaling:

	def __init__(self, target_feature, scaling_property_uri):
		"""
		Args:
			target_feature (str): URI of the geo feature class whose instances are used by the scaling process to generate observations.
			scaling_property_uri (str): URI identifying the observed property to scale.
		"""

		self.target_feature = target_feature
		self.scaling_property_uri = scaling_property_uri

		# fetching the procedure specifications
		self.fetch_scaling_procedure(target_feature)

		# Identify whether the scaling is upscaling or downscaling.
		self.scaling_type = self.find_scaling_type()


	def run_sparql_query(self, query):
		response = requests.get(SPARQL_ENDPOINT, params={'query': query, 'format': 'json'})
		return response.json()['results']['bindings']



	def SumAggregation(self, part_observations, whole_observation=None):
		"""
		Sample aggregation method for upscaling

		Args:
			part_observations: Dictionary of partial observations for constructing the whole-entity observation.
			whole_observation: Not relevant for this type of scaling method.

		Returns: 
			List of generated observations.
		"""

		items = part_observations.values()

		scaled_value = sum(item[0]["value"] for item in items)
		result_time = next(iter(items))[0]['resultTime']

		scaled_observation = f'''
		<id/observation/{uuid.uuid4().hex}> a sosa:Observation ;  
			sosa:resultTime "{result_time}"^^xsd:dateTime ; 
			sosa:hasFeatureOfInterest <{self.whole_entity}> ; 
			sosa:observedProperty <{self.scaling_property_uri}> ; 
			sosa:hasSimpleResult "{scaled_value}"^^xsd:integer ; 
			sosa:usedProcedure <{self.scaling_procedure}> . '''

		return [scaled_observation]



	def WeightedDisaggregation(self, part_observations, whole_observation):
		"""
		Sample weighted disaggregation method for downscaling

		Args:
			part_observations: Dictionary of partial observations for constructing the whole-entity observation.
			whole_observation: The whole-entity observation that should be disaggregated to generate part observations.

		Returns: 
			List of generated observations.
		"""

		total_whole_value = whole_observation[0]['value']
		result_time = whole_observation[0]['resultTime']

		items = part_observations.values()
		total_auxiliary = sum(item[0]["value"] for item in items)

		observations = []
		for part in part_observations:

			scaled_value = round(total_whole_value * (part_observations[part][0]['value'] / total_auxiliary))

			observations.append(f'''
				<id/observation/{uuid.uuid4().hex}> a sosa:Observation ;  
					sosa:resultTime "{result_time}"^^xsd:dateTime ; 
					sosa:hasFeatureOfInterest <{part}> ; 
					sosa:observedProperty <{self.scaling_property_uri}> ; 
					sosa:hasSimpleResult "{scaled_value}"^^xsd:integer ; 
					sosa:usedProcedure <{self.scaling_procedure}> . ''')
		return observations



	def run_scaling_method(self, function_name, parts, whole):
		func = getattr(self, function_name)
		return func(parts, whole)


	def get_feature_class(self):
		"""
		Queries the knowledge graph to determine the class name of the target region. According to the SOGO specification, this class should be a subclass of geo:Feature.

		Returns:
			URI of the target feature's class.
		"""
		query = f'''
		PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
		PREFIX geo: <http://www.opengis.net/ont/geosparql#>

		SELECT ?target_feature WHERE {{
			<{self.target_region_uri}> a ?target_feature .
			?target_feature rdfs:subClassOf geo:Feature .
		}}
		'''
		result = self.run_sparql_query(query)
		return result[0]['target_feature']['value'] if result else None
	

	def fetch_scaling_procedure(self, target_feature):
		"""
		Fetch the scaling procedure matching the given feature class from knowledge graph.
		TODO: Temporal extent should be parsed properly
		"""

		query = f'''
		PREFIX sogo: <{SOGO_NAMESPACE}>
		PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

		SELECT ?procedure ?mapping_query ?method ?auxiliary WHERE {{

		?procedure a sogo:ScalingProcedure ;
			sogo:hasScalingProperty <{self.scaling_property_uri}> ;
			sogo:usesScalingMethod ?method ;
			sogo:hasSpatialMap ?map .
			?map sogo:targetFeature <{target_feature}> ;
				sogo:mappingQuery ?mapping_query
			OPTIONAL {{
				?procedure sogo:hasAuxiliaryProperty ?auxiliary.
			}}
		}}'''	

		result = self.run_sparql_query(query)

		if not result:
			print("No scaling procedure found for the given property and target feature.")
			return

		self.scaling_procedure = result[0]['procedure']['value']
		self.scaling_method = result[0]['method']['value'].split('#')[-1]
		self.auxiliary_property_uri = result[0]['auxiliary']['value'] if 'auxiliary' in result[0] else None

		mapping_query = result[0]['mapping_query']['value'] 
		select_pos = re.search(r"\bSELECT\b", mapping_query, re.IGNORECASE).start()

		self.mapping_prefixes = mapping_query[:select_pos].strip()
		self.mapping_query = mapping_query[select_pos:].strip()

	
	def find_scaling_type(self):
		query = f'''
			PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
			PREFIX sogo: <{SOGO_NAMESPACE}>

			SELECT ?scaling_type WHERE {{
				<{SOGO_NAMESPACE}{self.scaling_method}> a ?scaling_type .
				?scaling_type rdfs:subClassOf sogo:ScalingMethod .
			}}
		'''
		result = self.run_sparql_query(query)
		scaling_type = mapping_query = result[0]['scaling_type']['value']

		return 	scaling_type.split('#')[-1]


	def get_whole_observation(self, whole_entity, observed_property, temporal_extent):
		"""
		Fetching whole-entity observation to generate the value for the target observation(s). 

		Args:
			whole_entity: URI of whole-entity.
			observed_property: URI identifying the observed property.
			temporal_extent: Temporal extent of observations.

		Retures:
			Dictionary of observation IDs and their corrsponding values.
		"""

		query = f'''

		PREFIX sosa:  <http://www.w3.org/ns/sosa/>

		SELECT ?property_value ?resultTime 
		WHERE {{
			?obs sosa:hasFeatureOfInterest <{whole_entity}> ;
				sosa:observedProperty <{observed_property}> ;
				sosa:hasSimpleResult ?property_value ;
				sosa:resultTime ?resultTime . 
				FILTER (YEAR(?resultTime) = {temporal_extent})
		}} ''' 

		results = self.run_sparql_query(query)

		observations = []

		for item in results:
			resultTime = item['resultTime']['value']
			value = item['property_value']['value']
			datatype = item['property_value']['datatype']
			
			converter = TYPE_CONVERTERS.get(datatype, str)
			converted_value = converter(value)
			
			observations.append({'value': converted_value, 'resultTime': resultTime})

		return observations



	def get_part_observations(self, whole_entity, observed_property, temporal_extent):
		"""
		Fetching part-entity observations to generate the observations. 

		Args:
			whole_entity: URI of whole-entity.
			observed_property: URI identifying the observed property.
			temporal_extent: Temporal extent of observations.

		Retures:
			Dictionary of observation IDs and their corrsponding values.
		"""

		mapping_query = self.mapping_query.replace("SOGO_WHOLE", whole_entity)
		mapping_query = mapping_query.replace("<SOGO_PART>", "?part")


		query = f'''

		
			PREFIX sosa:  <http://www.w3.org/ns/sosa/>
			{self.mapping_prefixes}

			SELECT ?part ?property_value ?resultTime 
			WHERE {{
				{{
					{mapping_query}
				}}

				?obs sosa:hasFeatureOfInterest ?part ;
					sosa:observedProperty <{observed_property}> ;
					sosa:hasSimpleResult ?property_value ;
					sosa:resultTime ?resultTime . 
					FILTER (YEAR(?resultTime) = {temporal_extent})

			}}  ''' 


		results = self.run_sparql_query(query)

		observations = dict()

		for item in results:
			part_uri = item['part']['value']
			resultTime = item['resultTime']['value']
			value = item['property_value']['value']
			datatype = item['property_value']['datatype']
			
			converter = TYPE_CONVERTERS.get(datatype, str)
			converted_value = converter(value)

			observation = {'value': converted_value, 'resultTime': resultTime}

			if part_uri in observations:
				observations[part_uri].append(observation)
			else:
				observations[part_uri] = [observation]

		return observations


	def generate_observations(self, whole_entity, temporal_extent):
		"""
		Generate an observation based on given region, property, and year.

		Args:
			whole_entity: URI of whole-entity.
			temporal_extent: Temporal extent of observations.


		Returns:
			List of generated observations.
		"""

		self.whole_entity = whole_entity

		if self.scaling_type == "UpscalingMethod":
			whole_observation = None
			part_observations = self.get_part_observations(whole_entity, self.scaling_property_uri, temporal_extent)
		else: 
			whole_observation = self.get_whole_observation(whole_entity, self.scaling_property_uri, temporal_extent)
			part_observations = self.get_part_observations(whole_entity, self.auxiliary_property_uri, temporal_extent)

		observations = self.run_scaling_method(self.scaling_method, part_observations, whole_observation)

		return observations




## Downscaling Example

In [101]:
sogo = SogoScaling(f"{CRISP_NAMESPACE}H3Cell", f"{CRISP_NAMESPACE}population")
observations = sogo.generate_observations("http://crispy.ai.wu.ac.at/crisp/id/community/92101", 2011)
for obs in observations:
    print(obs)



				<id/observation/90f4eaec9f354f529d04c48c4555bd9f> a sosa:Observation ;  
					sosa:resultTime "2011-10-31T00:00:00Z"^^xsd:dateTime ; 
					sosa:hasFeatureOfInterest <http://crispy.ai.wu.ac.at/crisp/id/cell/881e064c83fffff> ; 
					sosa:observedProperty <http://crispy.ai.wu.ac.at/crisp/population> ; 
					sosa:hasSimpleResult "857"^^xsd:integer ; 
					sosa:usedProcedure <http://crispy.ai.wu.ac.at/crisp/populationDownscaling> . 

				<id/observation/3fed515b2efb4f94be8a1b138bf0e0a7> a sosa:Observation ;  
					sosa:resultTime "2011-10-31T00:00:00Z"^^xsd:dateTime ; 
					sosa:hasFeatureOfInterest <http://crispy.ai.wu.ac.at/crisp/id/cell/881e064c87fffff> ; 
					sosa:observedProperty <http://crispy.ai.wu.ac.at/crisp/population> ; 
					sosa:hasSimpleResult "2954"^^xsd:integer ; 
					sosa:usedProcedure <http://crispy.ai.wu.ac.at/crisp/populationDownscaling> . 

				<id/observation/a295cfb73fe149a6a76cd358c63e5fd8> a sosa:Observation ;  
					sosa:resultTime "2011-10-31T00:00:00Z"^^xs

## Upscaling Example

In [102]:
sogo = SogoScaling(f"{CRISP_NAMESPACE}State", f"{CRISP_NAMESPACE}population")
obs = sogo.generate_observations("http://crispy.ai.wu.ac.at/crisp/id/state/1", 2011)
print(obs[0])


		<id/observation/29378083256e449da5ed9a64264ce64e> a sosa:Observation ;  
			sosa:resultTime "2011-10-31T00:00:00Z"^^xsd:dateTime ; 
			sosa:hasFeatureOfInterest <http://crispy.ai.wu.ac.at/crisp/id/state/1> ; 
			sosa:observedProperty <http://crispy.ai.wu.ac.at/crisp/population> ; 
			sosa:hasSimpleResult "285685"^^xsd:integer ; 
			sosa:usedProcedure <http://crispy.ai.wu.ac.at/crisp/populationUpscaling> . 
